# Looking at gridMET weather data

I want to see what weather files I downloaded, what variables are in them, and what the weather looks like over time. This is separate from LANDFIRE.

## 1. Import stuff

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Find the main project folder so the data paths still work from explorations/
PROJECT_ROOT = next((p for base in (Path.cwd(), Path.cwd().resolve()) for p in [base, *base.parents] if (p / "data" / "raw").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/raw. Put explorations/ inside your fire-prediction project.")
RAW = PROJECT_ROOT / "data" / "raw"
print("Project root:", PROJECT_ROOT)
print("Data folder:", RAW)
import xarray as xr
import numpy as np

## 2. Find my weather files

In [ ]:
combined = sorted(RAW.glob('gridmet_*.nc'))
yearly = sorted((RAW / 'gridmet_yearly').glob('*.nc'))
files = combined if combined else yearly
print(f'Combined files: {len(combined)}; yearly files: {len(yearly)}')
display(pd.DataFrame([{'path':str(p.relative_to(RAW)), 'size_MB':round(p.stat().st_size/1e6,2)} for p in files]))
if not files:
    raise FileNotFoundError('No gridMET .nc files found. Run your download_gridmet() script first.')

## 3. Open one file and see what is in it

In [ ]:
# Use the combined weather file if there is one
path = combined[-1] if combined else yearly[0]
ds = xr.open_dataset(path)
print('Opened:', path.relative_to(PROJECT_ROOT))
print(ds)
display(pd.DataFrame([{'variable':key, 'dims':str(val.dims), 'units':val.attrs.get('units', '(not specified)'), 'description':val.attrs.get('long_name', val.attrs.get('description',''))} for key,val in ds.data_vars.items()]))

## 4. What dates and locations does it cover?

In [ ]:
for axis in ('lat','lon','day','time'):
    if axis in ds.coords:
        a = ds[axis]
        print(axis, 'count:', a.size, 'first:', a.values[0], 'last:', a.values[-1])
print('Missing values by variable (sample first timestep only):')
for name, arr in ds.data_vars.items():
    if {'lat','lon'}.issubset(arr.dims):
        sample=arr.isel({d:0 for d in arr.dims if d not in ('lat','lon')})
        print(name, 'missing fraction:', round(float(sample.isnull().mean().compute()),3))

## 5. Make a weather map

In [ ]:
# Pick the first weather variable that can be mapped
candidates = [name for name,v in ds.data_vars.items() if {'lat','lon'}.issubset(v.dims)]
if not candidates:
    print('No lat/lon gridded variables found.')
else:
    name=candidates[0]
    a=ds[name]
    time_dim=next((d for d in ('day','time') if d in a.dims), None)
    if time_dim: a=a.isel({time_dim:0})
    a.plot(figsize=(11,6), robust=True)
    plt.title(f'{name}: first available day')
    plt.show()

## 6. Look at weather for one location

In [ ]:
# Try a location here (change this if it is outside the downloaded area)
LAT, LON = 40.0, -105.0
if not {'lat','lon'}.issubset(ds.coords):
    print('Coordinates lat/lon not available in this file.')
else:
    latmin,latmax=sorted((float(ds.lat.min()),float(ds.lat.max())))
    lonmin,lonmax=sorted((float(ds.lon.min()),float(ds.lon.max())))
    if not (latmin<=LAT<=latmax and lonmin<=LON<=lonmax):
        print(f'Point ({LAT},{LON}) outside available area: lat {latmin:.1f}..{latmax:.1f}, lon {lonmin:.1f}..{lonmax:.1f}')
    else:
        point=ds.sel(lat=LAT, lon=LON, method='nearest')
        vars_to_plot=[k for k,v in point.data_vars.items() if 'day' in v.dims or 'time' in v.dims]
        if vars_to_plot:
            point[vars_to_plot[0]].plot(figsize=(12,4))
            plt.title(f'{vars_to_plot[0]} near {LAT}, {LON}')
            plt.show()
        else: print('No time-varying variables in selected file.')

## 7. What if I only have yearly files?

In [ ]:
if not combined:
    print('Each cached annual file usually contains one weather variable for one year.')
    print('Example files:', [p.name for p in yearly[:8]])
    print('To access all 2020 variables together, open matching 2020 files with xarray and merge after checking their coordinates.')
else:
    print('You have a combined file: no merging is necessary for basic exploration.')
# Close file when finished using the notebook: ds.close()

## Notes

Need to check the units before using the data. For example, temperature might be in Kelvin. For the regression, weather should come from days before the fire, not afterward.